# VELALAR COLLEGE OF ENGINEERING AND TECHNOLOGY, ERODE - 12
## DEPARTMENT OF INFORMATION TECHNOLOGY
### ACADEMIC YEAR: 2026-27 (ODD) | CLASS: 25IT3A&B
### COURSE: FOUNDATIONS OF DATA SCIENCE (25ITC31)
### SCENARIO 8: STUDENT ACADEMIC PERFORMANCE AND RESULT ANALYSIS SYSTEM
**Student Name:** Jaswanth surya S  
**Register Number:** 732925ITR038  
**Class:** 25IT3A  

---
## 1. Project Objectives
1. Connect to the institutional academic SQLite database and extract marks, students, and curriculum records.
2. Perform data quality audit, missing value checks, and outlier detection using Pandas and NumPy.
3. Engineer academic performance metrics: GPA, classification tiers (Distinction, First Class, Remedial).
4. Generate exploratory data analysis (EDA) visualizations using Matplotlib and Seaborn.
5. Automatically detect subjects with low pass rates (< 85%) for remedial action.

In [ ]:
# Step 1: Import Essential Data Science Libraries (Unit 2 & 3)
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
print("Libraries successfully imported!")

In [ ]:
# Step 2: Database Retrieval via SQL Connection
conn = sqlite3.connect('../database/student_performance.db')

query = '''
SELECT 
    s.register_no,
    s.name AS student_name,
    s.section,
    d.dept_code,
    sub.subject_code,
    sub.subject_name,
    sub.credits,
    m.internal_marks,
    m.external_marks,
    m.total_marks,
    m.grade,
    m.grade_point,
    m.status
FROM marks m
JOIN students s ON m.register_no = s.register_no
JOIN subjects sub ON m.subject_code = sub.subject_code
JOIN departments d ON s.dept_id = d.dept_id;
'''

df_marks = pd.read_sql_query(query, conn)
df_students = pd.read_sql_query('SELECT * FROM v_student_semester_summary;', conn)
df_subjects = pd.read_sql_query('SELECT * FROM v_subject_performance;', conn)
conn.close()

print(f"Retrieved {len(df_marks)} marks records across {len(df_students)} students.")
df_marks.head(5)

In [ ]:
# Step 3: Data Quality Audit & Cleaning (Unit 4)
print("--- Missing Values Check ---")
print(df_marks.isnull().sum())

print("\n--- Duplicate Check ---")
print(f"Duplicates: {df_marks.duplicated().sum()}")

print("\n--- Summary Statistics ---")
display(df_marks[['internal_marks', 'external_marks', 'total_marks']].describe())

In [ ]:
# Step 4: Pivot Matrix - Students vs Subjects
pivot_matrix = df_marks.pivot_table(
    index=['register_no', 'student_name', 'section'],
    columns='subject_code',
    values='total_marks'
).reset_index()

display(pivot_matrix.head(10))

In [ ]:
# Step 5: Visualizing Subject-Wise Pass Percentage (Unit 5)
plt.figure(figsize=(10, 5))
colors = ['#d9534f' if p < 85 else '#5cb85c' for p in df_subjects['pass_percentage']]
plt.bar(df_subjects['subject_code'], df_subjects['pass_percentage'], color=colors, edgecolor='black')
plt.axhline(85, color='darkred', linestyle='--', label='Institutional Target (85%)')
plt.title('Subject-Wise Pass Rate & Low Pass Rate Identification', fontweight='bold')
plt.ylabel('Pass Rate (%)')
plt.legend()
plt.show()

In [ ]:
# Step 6: Grade Distribution & Correlation Analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Grade Countplot
sns.countplot(ax=axes[0], data=df_marks, x='grade', order=['O', 'A+', 'A', 'B+', 'B', 'RA'], palette='viridis')
axes[0].set_title('Letter Grade Distribution', fontweight='bold')

# Correlation Heatmap
sns.heatmap(ax=axes[1], data=df_marks[['internal_marks', 'external_marks', 'total_marks', 'grade_point']].corr(), annot=True, cmap='Blues')
axes[1].set_title('Assessment Correlation Matrix', fontweight='bold')

plt.tight_layout()
plt.show()